In [1]:
import pandas as pd
import random
from math import pow
from decimal import Decimal
import os
import pickle
os.chdir('Resources/')

In [2]:
def gcd(a, b):
    if a < b:
        return gcd(b, a)
    elif a % b == 0:
        return b
    else:
        return gcd(b, a % b)

In [3]:
def gen_key(q):
    key = random.randint(int(pow(10, 20)), q)
    while gcd(q, key) != 1:
        key = random.randint(int(pow(10, 20)), q)
    return key

In [4]:
def power(a, b, c):
    x = 1
    y = a
    while b > 0:
        if b % 2 != 0:
            x = (x * y) % c
        y = (y * y) % c
        b = int(b / 2)
    return x % c

In [5]:
def encrypt(value, q, h, g, fixed_key):
    s = power(h, fixed_key, q)
    p = power(g, fixed_key, q)
    return s * value, p

In [6]:
df = pd.read_csv('1_Structured_Data.csv')
X = df.drop(['HeartDisease'], axis='columns')
Y = df[['HeartDisease']]

In [7]:
q = random.randint(int(pow(10, 20)), int(pow(10, 50)))
g = random.randint(2, q)
fixed_key = gen_key(q)
h = power(g, fixed_key, q)

In [8]:
column_keys = {}
for column in X.columns:
    q = random.randint(int(pow(10, 20)), int(pow(10, 50)))
    g = random.randint(2, q)
    fixed_key = gen_key(q)
    h = power(g, fixed_key, q)
    column_keys[column] = (q, h, g, fixed_key)

encrypted_X = X.copy()
for column in X.columns:
    q, h, g, fixed_key = column_keys[column]
    encrypted_values = []
    for value in X[column]:
        if isinstance(value, str):
            numeric_value = sum(ord(char) for char in value)
            encrypted_value, _ = encrypt(numeric_value, q, h, g, fixed_key)
        else:
            encrypted_value, _ = encrypt(float(value), q, h, g, fixed_key)
        encrypted_values.append(encrypted_value)
    encrypted_X[column] = encrypted_values

In [9]:
Y['HeartDisease'] = Y['HeartDisease'].replace({'No': 0, 'Yes': 1})

C:\Users\USER\AppData\Local\Temp\ipykernel_5240\2653026528.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Y['HeartDisease'] = Y['HeartDisease'].replace({'No': 0, 'Yes': 1})
C:\Users\USER\AppData\Local\Temp\ipykernel_5240\2653026528.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Y['HeartDisease'] = Y['HeartDisease'].replace({'No': 0, 'Yes': 1})


In [10]:
encrypted_df = pd.concat([encrypted_X, Y], axis=1)
encrypted_df.to_csv('3_Encrypted_Data.csv', index=False)

In [11]:
public_key = (q, h, g, fixed_key)
public_key_file_path = '3_Public_Key.pkl'

with open(public_key_file_path, 'wb') as public_key_file:
    pickle.dump(column_keys, public_key_file)

In [12]:
import pickle

# Define the file path where the public keys are saved
public_key_file_path = '3_Public_Key.pkl'

# Load the encryption keys and parameters from the file
with open(public_key_file_path, 'rb') as public_key_file:
    column_keys = pickle.load(public_key_file)

# Now, column_keys contains the encryption keys and parameters for each column
# You can access them as follows:
print(column_keys['Age'])  # Example: Print the encryption key and parameters for the 'Age' column


(87670266624105525429483067119323668684127738971229, 86005389665682172821959314123761906950841593890595, 23170942030581678516732287605837538134633662918161, 55932448984661509113252988817756371481312519153501)
